In [1]:
# Random sampling: CART vs DeepSeek-R1
from pathlib import Path
from io import BytesIO
import zipfile
import pandas as pd
import requests

from IPython.display import display

SAMPLE_SIZE = 100
RANDOM_SEED = 42

EXPECTED_ROWS = 3656

GITHUB_BASE = (
    "https://raw.githubusercontent.com/semoglou/cart/"
    "main/llm_baselines/evaluation/"
)

ZIP_URL = (
    GITHUB_BASE
    + "evaluation_cart_deepseek_prepared.zip"
)

INPUT_FILENAME = (
    "evaluation_input_cart_deepseek.csv"
)

MAPPING_FILENAME = (
    "evaluation_mapping_cart_deepseek.csv"
)

SAMPLE_DIR = (
    Path("/content/samples")
    / f"sample_{SAMPLE_SIZE}"
)

SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Source: {ZIP_URL}")

response = requests.get(
    ZIP_URL,
    timeout=60
)

response.raise_for_status()

with zipfile.ZipFile(
    BytesIO(response.content)
) as zip_file:

    available_files = zip_file.namelist()

    print("\nFiles inside ZIP:")

    for filename in available_files:
        print(f" - {filename}")

    assert INPUT_FILENAME in available_files, (
        f"Missing {INPUT_FILENAME}"
    )

    assert MAPPING_FILENAME in available_files, (
        f"Missing {MAPPING_FILENAME}"
    )

    with zip_file.open(INPUT_FILENAME) as f:

        evaluation_input = pd.read_csv(
            f,
            encoding="utf-8-sig"
        )

    with zip_file.open(MAPPING_FILENAME) as f:

        evaluation_mapping = pd.read_csv(
            f,
            encoding="utf-8-sig",
            dtype={"paragraph_id": "string"}
        )

print("\n" + "=" * 60)
print("VALIDATING SOURCE DATASETS")
print("=" * 60)

assert len(evaluation_input) == EXPECTED_ROWS, (
    "Unexpected evaluation input size."
)

assert len(evaluation_mapping) == EXPECTED_ROWS, (
    "Unexpected evaluation mapping size."
)

required_input_columns = [
    "annotation_id",
    "paragraph",
    "option_A",
    "option_B",
    "option_C"
]

required_mapping_columns = [
    "annotation_id",
    "paragraph_id",
    "system_A",
    "system_B"
]

assert all(
    col in evaluation_input.columns
    for col in required_input_columns
), "Missing columns in evaluation input."

assert all(
    col in evaluation_mapping.columns
    for col in required_mapping_columns
), "Missing columns in evaluation mapping."

assert evaluation_input["annotation_id"].is_unique

assert evaluation_mapping["annotation_id"].is_unique

assert evaluation_mapping["paragraph_id"].is_unique

assert evaluation_input.notna().all().all()

assert evaluation_mapping.notna().all().all()

assert (
    set(evaluation_input["annotation_id"])
    ==
    set(evaluation_mapping["annotation_id"])
), "Annotation IDs do not match."

assert (
    evaluation_input["option_C"]
    .eq("None of the above")
    .all()
), "Unexpected option C values."

assert (
    evaluation_mapping[["system_A", "system_B"]]
    .apply(
        lambda row: set(row) == {
            "CART",
            "DeepSeek-R1"
        },
        axis=1
    )
    .all()
), "Unexpected system mapping."

assert SAMPLE_SIZE <= len(evaluation_input)

print(f"Evaluation input rows: {len(evaluation_input):,}")
print(f"Evaluation mapping rows: {len(evaluation_mapping):,}")

print("\nSource validation passed.")

sampled_input = (
    evaluation_input
    .sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

sampled_mapping = (
    evaluation_mapping
    .set_index("annotation_id")
    .loc[sampled_input["annotation_id"]]
    .reset_index()
)

assert len(sampled_input) == SAMPLE_SIZE

assert len(sampled_mapping) == SAMPLE_SIZE

assert sampled_input["annotation_id"].is_unique

assert sampled_mapping["annotation_id"].is_unique

assert (
    sampled_input["annotation_id"].tolist()
    ==
    sampled_mapping["annotation_id"].tolist()
), "Sample mapping mismatch."

assert (
    sampled_mapping[["system_A", "system_B"]]
    .apply(
        lambda row: set(row) == {
            "CART",
            "DeepSeek-R1"
        },
        axis=1
    )
    .all()
)

input_path = (
    SAMPLE_DIR
    / f"evaluation_input_sample{SAMPLE_SIZE}.csv"
)

mapping_path = (
    SAMPLE_DIR
    / f"evaluation_mapping_sample{SAMPLE_SIZE}.csv"
)

sampled_input.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

sampled_mapping.to_csv(
    mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Original dataset: {len(evaluation_input):,}")
print(f"Sample size: {len(sampled_input)}")
print(f"Random seed: {RANDOM_SEED}")

print(
    f"Unique annotation IDs: "
    f"{sampled_input['annotation_id'].nunique()}"
)

print(
    f"Unique paragraph IDs: "
    f"{sampled_mapping['paragraph_id'].nunique()}"
)

print("\nSaved files:")

print(f"Input:   {input_path}")
print(f"Mapping: {mapping_path}")

print("\nSystem distribution in sample:")

print(
    sampled_mapping["system_A"].value_counts()
)

print("\nSampling validation passed.")

Source: https://raw.githubusercontent.com/semoglou/cart/main/llm_baselines/evaluation/evaluation_cart_deepseek_prepared.zip

Files inside ZIP:
 - evaluation_input_cart_deepseek.csv
 - evaluation_mapping_cart_deepseek.csv

VALIDATING SOURCE DATASETS
Evaluation input rows: 3,656
Evaluation mapping rows: 3,656

Source validation passed.
Original dataset: 3,656
Sample size: 100
Random seed: 42
Unique annotation IDs: 100
Unique paragraph IDs: 100

Saved files:
Input:   /content/samples/sample_100/evaluation_input_sample100.csv
Mapping: /content/samples/sample_100/evaluation_mapping_sample100.csv

System distribution in sample:
system_A
CART           53
DeepSeek-R1    47
Name: count, dtype: int64

Sampling validation passed.


In [2]:
display(sampled_input.head())

,annotation_id,paragraph,option_A,option_B,option_C
0,496,Συνεπώς είναι προφανές πως αν λειτουργούσε αυτ...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation,None of the above
1,1133,Γιατί το δίλημμα «τα κέρδη ή οι ζωές» αποκαλύπ...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation,None of the above
2,1002,"Ο κάθε Βουλευτής των υπολοίπων κομμάτων, και κ...",Legal and Institutional Reform Imperatives,Systemic governance failures caused preventabl...,None of the above
3,3164,Εμείς λέμε «όχι» σε αυτό το μοντέλο των δήθεν ...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation,None of the above
4,1593,"Επόμενο, διότι η στρατηγική και η πολιτική των...",Railway privatization caused systemic safety f...,Systemic Governance Negligence Causation,None of the above


In [3]:
display(sampled_mapping.head())

,annotation_id,paragraph_id,system_A,system_B
0,496,2023-11-24_84_102_p5,CART,DeepSeek-R1
1,1133,2024-03-27_149_129_p5,CART,DeepSeek-R1
2,1002,2024-03-20_144_73_p16,DeepSeek-R1,CART
3,3164,2025-06-10_360_136_p5,CART,DeepSeek-R1
4,1593,2024-10-17_235_291_p5,CART,DeepSeek-R1


In [4]:
from google.colab import files
files.download(str(input_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from google.colab import files
files.download(str(mapping_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>